# SelvaSonic — Entrenamiento Attention V4 (Semana 4)

## Objetivo

Entrenar el modelo **SelvaSonicCNNAttention** (CNN + Multi-Head Self-Attention) bajo las mismas condiciones que el baseline, para una comparación honesta.

## Cambios respecto al V3 (baseline)

1. **Modelo:** `SelvaSonicCNNAttention` en lugar de `SelvaSonicCNN` (4 heads, PE aprendible, +69% params)
2. **TensorBoard desde el inicio:** usa `TrainingLogger` (no conversión posterior)
3. **Matriz de confusión cada 5 épocas:** logueada a TensorBoard
4. **Run name diferente:** `attention_S4_v1_<timestamp>` para no confundir con baseline
5. **Comparación al final:** delta de val_acc/test_acc vs baseline

## Hipótesis a verificar

Del análisis de errores del baseline (notebook 04), esperamos que el attention:
- Mejore la discriminación entre especies del mismo género (`Crypturellus_*`).
- Reduzca la fuga hacia `no_ave` en clips con poco canto.
- Suba el **macro F1** (más que el accuracy global).

## Celda 1 — Montar Drive y verificar GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("NO HAY GPU. Cambia el runtime a T4 GPU")

## Celda 2 — Descomprimir código y datos

In [ ]:
import os
import zipfile
import shutil

PROYECTO = '/content/SelvaSonic-ML'
DRIVE_BASE = '/content/drive/MyDrive/SelvaSonic_Proyecto'

# IMPORTANTE: si vas a reentrenar y el zip cambio (nueva version con attention),
# debes borrar la carpeta vieja primero para que se vuelva a descomprimir.
# Descomenta la siguiente linea SOLO la primera vez que ejecutes este notebook:
FORZAR_REDESCOMPRESION = True  # <-- ponlo en True la primera vez

if FORZAR_REDESCOMPRESION and os.path.exists(PROYECTO):
    print(f"Borrando {PROYECTO} para re-descomprimir version con attention...")
    shutil.rmtree(PROYECTO)

if os.path.exists(os.path.join(PROYECTO, 'src', 'model.py')) and \
   os.path.exists(os.path.join(PROYECTO, 'data', 'raw')):
    print("OK Proyecto ya restaurado, saltando descompresion")
else:
    os.makedirs(PROYECTO, exist_ok=True)

    print("[1/3] Descomprimiendo codigo (version S4 con attention)...")
    with zipfile.ZipFile(f'{DRIVE_BASE}/codigo.zip', 'r') as z:
        z.extractall(PROYECTO)

    print("[2/3] Descomprimiendo datos (puede tardar 2-3 min)...")
    os.makedirs(f'{PROYECTO}/data/raw', exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/data_raw.zip', 'r') as z:
        z.extractall(f'{PROYECTO}/data/raw')

    print("[3/3] Fix de backslash en nombres (bug de zip de Windows)...")
    for base in [PROYECTO, f'{PROYECTO}/data/raw']:
        archivos_con_bs = [i for i in os.listdir(base) if '\\' in i]
        for nombre_viejo in archivos_con_bs:
            nombre_nuevo = nombre_viejo.replace('\\', '/')
            ruta_vieja = os.path.join(base, nombre_viejo)
            ruta_nueva = os.path.join(base, nombre_nuevo)
            os.makedirs(os.path.dirname(ruta_nueva), exist_ok=True)
            if os.path.isdir(ruta_vieja):
                if os.path.exists(ruta_nueva):
                    shutil.rmtree(ruta_nueva)
                shutil.move(ruta_vieja, ruta_nueva)
            else:
                shutil.move(ruta_vieja, ruta_nueva)

    print("OK Proyecto y datos listos")

os.chdir(PROYECTO)
print(f"\nDirectorio actual: {os.getcwd()}")

# Verificar que la nueva clase Attention existe
with open(f'{PROYECTO}/src/model.py') as f:
    contenido = f.read()
if 'SelvaSonicCNNAttention' in contenido:
    print("OK SelvaSonicCNNAttention detectada en model.py")
else:
    raise RuntimeError("ERROR: SelvaSonicCNNAttention NO esta en model.py. Re-sube codigo.zip a Drive.")

## Celda 3 — Instalar dependencias

In [ ]:
!pip install -q librosa==0.10.1 soundfile pyyaml tensorboard
print("OK dependencias instaladas")

## Celda 4 — Importar módulos (con TrainingLogger)

In [ ]:
import sys
sys.path.insert(0, '/content/SelvaSonic-ML')

from src.dataset import create_dataloaders
from src.model import SelvaSonicCNNAttention  # <-- el nuevo modelo
from src.logger import TrainingLogger        # <-- TensorBoard desde el inicio
from src import config

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import json
import time
from datetime import datetime

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("OK Modulos importados")

## Celda 5 — Configuración del entrenamiento

**Mantenemos los mismos hiperparámetros del baseline** para que la comparación sea limpia. Solo cambia el modelo.

In [ ]:
# IDENTICOS al baseline (para comparacion honesta)
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_T_MAX = EPOCHS
NUM_WORKERS = 2
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Cada cuantas epocas loguear matriz de confusion a TensorBoard
LOG_CM_EVERY = config.LOG_CONFUSION_MATRIX_EVERY_N_EPOCHS  # 5 por defecto

# Run name: distinto del baseline para no confundir
RUN_NAME = f"attention_S4_v1_{datetime.now().strftime('%Y%m%d_%H%M')}"
DRIVE_RUN_DIR = f'{DRIVE_BASE}/runs/{RUN_NAME}'
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

# Carpeta de checkpoint persistente (separada del baseline)
DRIVE_CKPT_DIR = f'{DRIVE_BASE}/checkpoints_activos_attention'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
LATEST_CKPT = f'{DRIVE_CKPT_DIR}/latest.pth'
BEST_CKPT = f'{DRIVE_CKPT_DIR}/best.pth'
HISTORY_JSON = f'{DRIVE_CKPT_DIR}/history.json'

# TensorBoard log dir (vive dentro del run dir, en Drive)
TB_LOG_DIR = f'{DRIVE_RUN_DIR}/tensorboard'

print(f"Run name: {RUN_NAME}")
print(f"Run dir:  {DRIVE_RUN_DIR}")
print(f"TB dir:   {TB_LOG_DIR}")
print(f"\nHiperparametros (identicos al baseline):")
print(f"  EPOCHS:           {EPOCHS}")
print(f"  BATCH_SIZE:       {BATCH_SIZE}")
print(f"  LR:               {LEARNING_RATE}")
print(f"  WEIGHT_DECAY:     {WEIGHT_DECAY}")
print(f"  LABEL_SMOOTHING:  {LABEL_SMOOTHING}")
print(f"  PATIENCE:         {EARLY_STOPPING_PATIENCE}")
print(f"  LOG_CM_EVERY:     {LOG_CM_EVERY} epocas")

## Celda 6 — DataLoaders

In [ ]:
RAW_DATA_DIR = f'{PROYECTO}/data/raw'

train_loader, val_loader, test_loader, label_map = create_dataloaders(
    raw_data_dir=RAW_DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    random_state=SEED,    # MISMA semilla del baseline -> mismo split
    verbose=True,
)

NUM_CLASSES = len(label_map)
idx_to_name = {v: k for k, v in label_map.items()}
class_names = [idx_to_name[i] for i in range(NUM_CLASSES)]

print(f"\nTrain: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Clases ({NUM_CLASSES}): {class_names}")

# Smoke test
x_sample, y_sample = next(iter(train_loader))
print(f"\nBatch shape: {tuple(x_sample.shape)}")

## Celda 7 — Modelo con Attention + loss + optimizer + scheduler

In [ ]:
model = SelvaSonicCNNAttention(num_classes=NUM_CLASSES).to(device)
n_params = model.count_parameters()
print(f"Modelo: SelvaSonicCNNAttention")
print(f"Parametros entrenables: {n_params:,}")
print(f"  Baseline tenia:       422,635")
print(f"  Diferencia:           +{n_params - 422635:,} (+{(n_params/422635 - 1)*100:.1f}%)")

# Smoke test forward
model.eval()
with torch.no_grad():
    out_test = model(x_sample.to(device))
assert out_test.shape == (BATCH_SIZE, NUM_CLASSES), f"Shape incorrecto: {out_test.shape}"
print(f"\nOK Forward: {tuple(x_sample.shape)} -> {tuple(out_test.shape)}")

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULER_T_MAX, eta_min=1e-6)

print(f"\nLoss:      CrossEntropyLoss(label_smoothing={LABEL_SMOOTHING})")
print(f"Optimizer: AdamW(lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: CosineAnnealingLR(T_max={SCHEDULER_T_MAX})")

## Celda 8 — Inicializar TrainingLogger (TensorBoard)

In [ ]:
hparams_dict = {
    'model': 'SelvaSonicCNNAttention',
    'num_heads': config.ATTENTION_NUM_HEADS,
    'attention_dropout': config.ATTENTION_DROPOUT,
    'classifier_dropout': config.DROPOUT,
    'batch_size': BATCH_SIZE,
    'lr': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'epochs_max': EPOCHS,
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'n_params': n_params,
}

tb_logger = TrainingLogger(
    log_dir=TB_LOG_DIR,
    run_name=RUN_NAME,
    label_map=label_map,
    hparams=hparams_dict,
    enabled=True,
)

# Intentar loguear el grafo del modelo (puede fallar con MultiheadAttention, no es critico)
try:
    tb_logger.log_model_graph(model, x_sample.to(device))
    print("OK Grafo del modelo logueado")
except Exception as e:
    print(f"AVISO: no se pudo loguear el grafo ({e}). No es critico.")

print(f"OK TensorBoard inicializado en: {TB_LOG_DIR}")

## Celda 9 — Funciones auxiliares de checkpoint + entrenamiento + eval

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, epoch, history, best_val_acc, label_map):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_val_acc': best_val_acc,
        'label_map': label_map,
        'hparams': hparams_dict,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    return ckpt['epoch'] + 1, ckpt['history'], ckpt['best_val_acc']


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_y, all_pred = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
        if return_preds:
            all_y.append(y.cpu())
            all_pred.append(pred.cpu())
    if return_preds:
        return total_loss / total, correct / total, torch.cat(all_y).numpy(), torch.cat(all_pred).numpy()
    return total_loss / total, correct / total


print("OK Funciones definidas")

## Celda 10 — Resume automático

In [ ]:
start_epoch = 0
best_val_acc = 0.0
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': [], 'epoch_time_s': [],
}

RESUME = True

if RESUME and os.path.exists(LATEST_CKPT):
    print(f"Checkpoint encontrado en {LATEST_CKPT}")
    start_epoch, history, best_val_acc = load_checkpoint(
        LATEST_CKPT, model, optimizer, scheduler
    )
    print(f"OK Reanudando desde epoca {start_epoch} (best_val_acc previo: {best_val_acc:.4f})")
else:
    if not RESUME:
        for f in [LATEST_CKPT, BEST_CKPT, HISTORY_JSON]:
            if os.path.exists(f):
                os.remove(f)
        print("RESUME=False, empezando de cero")
    else:
        print("No hay checkpoint previo, empezando de cero (epoca 0)")

## Celda 11 — 🚂 BUCLE DE ENTRENAMIENTO con TensorBoard

Misma lógica robusta del baseline (checkpoint a Drive cada época, resume automático), pero con:
- Log de scalars a TensorBoard cada época
- Log de matriz de confusión a TensorBoard cada `LOG_CM_EVERY` épocas

In [ ]:
print("=" * 75)
print(f"ENTRENAMIENTO ATTENTION — {RUN_NAME}")
print(f"Desde epoca {start_epoch} hasta {EPOCHS}")
print("=" * 75)

epochs_without_improvement = 0

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

    # Eval: en epocas de matriz de confusion, pedimos predicciones tambien
    if (epoch + 1) % LOG_CM_EVERY == 0:
        val_loss, val_acc, y_val_true, y_val_pred = evaluate(
            model, val_loader, criterion, device, return_preds=True
        )
    else:
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        y_val_true, y_val_pred = None, None

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    history['epoch_time_s'].append(elapsed)

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    marker = ' [BEST]' if is_best else ''
    print(
        f"Epoch {epoch+1:3d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f} acc={val_acc:.3f} | "
        f"lr={current_lr:.2e} | {elapsed:.1f}s{marker}"
    )

    # --- Log a TensorBoard ---
    tb_logger.log_epoch(
        epoch=epoch,
        train_loss=train_loss,
        train_acc=train_acc,
        val_loss=val_loss,
        val_acc=val_acc,
        lr=current_lr,
    )

    # Matriz de confusion cada N epocas
    if y_val_true is not None:
        try:
            tb_logger.log_confusion_matrix(epoch, y_val_true, y_val_pred, class_names)
        except Exception as e:
            print(f"  (aviso: no se pudo loguear CM esta epoca: {e})")

    # --- Checkpoint a Drive ---
    save_checkpoint(LATEST_CKPT, model, optimizer, scheduler, epoch, history, best_val_acc, label_map)
    if is_best:
        save_checkpoint(BEST_CKPT, model, optimizer, scheduler, epoch, history, best_val_acc, label_map)

    with open(HISTORY_JSON, 'w') as f:
        json.dump({
            'history': history,
            'best_val_acc': best_val_acc,
            'last_epoch_completed': epoch,
            'run_name': RUN_NAME,
            'label_map': label_map,
        }, f, indent=2)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\n[EARLY STOPPING] Val_acc no mejora en {EARLY_STOPPING_PATIENCE} epocas. Parando.")
        break

print("\n" + "=" * 75)
print(f"ENTRENAMIENTO TERMINADO. Mejor val_acc: {best_val_acc:.4f}")
print("=" * 75)

## Celda 12 — Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

COLOR_TRAIN = '#6C5CE7'
COLOR_VAL = '#00CEC9'
COLOR_LR = '#FD79A8'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#FAFAFA')
epochs_ran = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_ran, history['train_loss'], color=COLOR_TRAIN, lw=2, label='Train')
axes[0].plot(epochs_ran, history['val_loss'], color=COLOR_VAL, lw=2, label='Val')
axes[0].set_title('Pérdida (Attention)', fontsize=12); axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], color=COLOR_TRAIN, lw=2, label='Train')
axes[1].plot(epochs_ran, history['val_acc'], color=COLOR_VAL, lw=2, label='Val')
axes[1].axhline(best_val_acc, color='#2D3436', ls='--', alpha=0.5,
                label=f'Best val_acc = {best_val_acc:.3f}')
axes[1].set_title('Accuracy (Attention)', fontsize=12); axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history['lr'], color=COLOR_LR, lw=2)
axes[2].set_title('Learning Rate', fontsize=12); axes[2].set_xlabel('Época'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
fig_path = f'{DRIVE_RUN_DIR}/curvas_entrenamiento.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f"OK Figura guardada en {fig_path}")
plt.show()

## Celda 13 — Evaluar en test set con el mejor modelo

In [ ]:
print("Cargando best.pth para evaluacion en test...")
best_ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f"Cargado: epoca {best_ckpt['epoch']+1}, val_acc={best_ckpt['best_val_acc']:.4f}")

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\nTest loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")

## Celda 14 — Classification report + confusion matrix finales

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(y.numpy().tolist())

print("=" * 70)
print("CLASSIFICATION REPORT (test set) — ATTENTION MODEL")
print("=" * 70)
report = classification_report(
    all_labels, all_preds, target_names=class_names, digits=3, zero_division=0
)
print(report)

with open(f'{DRIVE_RUN_DIR}/classification_report.txt', 'w') as f:
    f.write(report)

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#FAFAFA')
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=class_names, yticklabels=class_names,
    cbar_kws={'label': 'Conteo'}, ax=ax,
)
ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real')
ax.set_title(f'Matriz de Confusión — Attention Test (acc={test_acc:.3f})')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
cm_path = f'{DRIVE_RUN_DIR}/confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
print(f"\nOK Matriz de confusion guardada en {cm_path}")
plt.show()

## Celda 15 — Comparación directa con el baseline

In [ ]:
# Resultados del baseline (de tu run anterior)
BASELINE = {
    'val_acc': 0.7419,
    'test_acc': 0.6322,
    'n_params': 422_635,
    'run_name': 'baseline_S3_v2_20260527_0118',
}

ATTN = {
    'val_acc': best_val_acc,
    'test_acc': test_acc,
    'n_params': n_params,
    'run_name': RUN_NAME,
}

delta_val = ATTN['val_acc'] - BASELINE['val_acc']
delta_test = ATTN['test_acc'] - BASELINE['test_acc']
delta_params = ATTN['n_params'] - BASELINE['n_params']

print("=" * 70)
print("COMPARACION BASELINE vs ATTENTION")
print("=" * 70)
print(f"{'Metrica':<20} {'Baseline':>12} {'Attention':>12} {'Delta':>10}")
print('-' * 60)
print(f"{'val_acc':<20} {BASELINE['val_acc']:>12.4f} {ATTN['val_acc']:>12.4f} {delta_val:>+10.4f}")
print(f"{'test_acc':<20} {BASELINE['test_acc']:>12.4f} {ATTN['test_acc']:>12.4f} {delta_test:>+10.4f}")
print(f"{'parametros':<20} {BASELINE['n_params']:>12,} {ATTN['n_params']:>12,} {delta_params:>+10,}")
print("=" * 70)

if delta_test > 0:
    print(f"\nEL ATTENTION MEJORA TEST_ACC EN +{delta_test:.4f} ({delta_test/BASELINE['test_acc']*100:+.1f}%)")
else:
    print(f"\nEL ATTENTION NO MEJORA EL BASELINE EN TEST (delta={delta_test:+.4f})")
    print("   Esto puede deberse a: overfitting por mas parametros, o necesidad")
    print("   de mas datos / class weights / data augmentation mas agresivo.")

# Guardar resumen final
summary = {
    'run_name': RUN_NAME,
    'timestamp': datetime.now().isoformat(),
    'epochs_completed': len(history['train_loss']),
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_loss': test_loss,
    'n_params': n_params,
    'hparams': hparams_dict,
    'comparison_baseline': BASELINE,
    'deltas': {
        'val_acc': delta_val,
        'test_acc': delta_test,
        'params': delta_params,
    },
}
with open(f'{DRIVE_RUN_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Snapshot final
shutil.copy(BEST_CKPT, f'{DRIVE_RUN_DIR}/best.pth')
shutil.copy(LATEST_CKPT, f'{DRIVE_RUN_DIR}/latest.pth')
shutil.copy(HISTORY_JSON, f'{DRIVE_RUN_DIR}/history.json')

# Cerrar TensorBoard logger
tb_logger.log_hparams_final({'best_val_acc': best_val_acc, 'test_acc': test_acc})
tb_logger.close()

print(f"\nOK Run completo guardado en {DRIVE_RUN_DIR}")
print("\nArchivos:")
for f in sorted(os.listdir(DRIVE_RUN_DIR)):
    size_mb = os.path.getsize(os.path.join(DRIVE_RUN_DIR, f)) / (1024 * 1024) if os.path.isfile(os.path.join(DRIVE_RUN_DIR, f)) else 0
    if size_mb > 0:
        print(f"  - {f}  ({size_mb:.2f} MB)")
    else:
        print(f"  - {f}/  (carpeta)")